# Array multinode  *(Capstone — the most meaningful exercise of the session)*

This notebook scales the same computation to **multiple Expanse nodes**. It is the most meaningful exercise of the session — the same Dask array call now spans a real distributed cluster.

**This notebook requires setup that the earlier notebooks do not.** You need a running Dask scheduler on this notebook node and a separate Dask worker job submitted from the login node. The cell below checks for the scheduler and tells you exactly what to do if it is not running yet.

## 0. Prerequisite — start the Dask cluster

Dask distributed needs two pieces:

1. A **scheduler** running on *this* notebook node (so the notebook can connect to it over localhost).
2. A **worker job** submitted from the *login node* that connects back to the scheduler.

### Start the scheduler (JupyterLab terminal, run once)

Open a terminal inside JupyterLab (or use the OnDemand terminal) and run:

```bash
bash dask_slurm/launch_scheduler.sh
```

Leave that terminal open — the scheduler runs in the foreground and prints logs as workers connect.

### Submit the worker job (login node terminal)

From a terminal on the Expanse login node, submit the pre-written worker script:

```bash
sbatch dask_slurm/dask_workers.slrm
```

Watch the scheduler terminal — within a minute or two you should see the workers from the worker nodes connect. Once they do, come back here.

In [ ]:
import socket
import time
from distributed import Client

address = socket.gethostbyname(socket.gethostname())
print(f"This notebook node address: {address}")

# Try to connect to a scheduler that should already be running on this node.
# We give it a few retries so participants who just started the scheduler
# can re-run this cell as it comes up.
client = None
for attempt in range(5):
    try:
        client = Client(f'{address}:8786', timeout='5s', set_as_default=True)
        client.wait_for_workers(1, timeout='10s')
        break
    except Exception as e:
        print(f"  attempt {attempt+1}/5: {type(e).__name__}: {e}")
        time.sleep(3)

if client is None:
    raise RuntimeError(
        "Could not connect to a Dask scheduler on this node.\n"
        "Start one first:  bash dask_slurm/launch_scheduler.sh\n"
        "and submit workers from the login node:  sbatch dask_slurm/dask_workers.slrm\n"
        "Then re-run this cell."
    )

print(f"Connected to scheduler: {client.scheduler.address}")
print(f"Number of workers connected: {len(client.scheduler_info()['workers'])}")

Connect to the client using the "Dask Lab Extension" (red/orange icon on the far left of the JupyterLab screen) using `/proxy/8787/status` as the address.

<a href="/proxy/22222/status" target="_blank">Click here to access the Dask Dashboard directly using the Jupyter Server Proxy extension</a>

## 1. Build a distributed array and compute

The API is identical to the single-machine Dask arrays from the previous notebook — the only difference is that `compute()` now ships the work to the worker job's nodes.

In [ ]:
import numpy as np
import dask.array as da

# A 200k x 200k array of ones, chunked into 10k x 10k blocks.
AA = da.ones((200000, 200000), chunks=(10000, 10000))
print(f"Number of blocks: {AA.numblocks}")
print(f"Array size on disk if materialized: {AA.nbytes / 1024**3:.1f} GB")

In [ ]:
# Sum every element, distributed across the worker nodes.
# The Dask dashboard (link above) shows the task graph executing across workers.
%time AA.sum().compute()

## 2. Take this home

The same pattern works on your own Expanse allocation:

1. Copy `dask_slurm/` into your project.
2. Edit `dask_workers.slrm` to use **your** `--account` and `--reservation` (the SI26 values are placeholders).
3. Start the scheduler on your notebook node, submit the worker job from the login node, and connect with `Client(...)`.

The key idea: **the Dask array API does not change when you go from one core to many cores to many nodes.** You write the same array expression; Dask ships the chunks to whatever workers are connected.

## 3. Take this home

The same pattern works on your own Expanse allocation:

1. Copy `dask_slurm/` into your project.
2. Edit `dask_workers.slrm` to use **your** `--account` and `--reservation` (the SI26 values are placeholders).
3. Start the scheduler on your notebook node, submit the worker job from the login node, and connect with `Client(...)`.

The key idea: **the Dask array API does not change when you go from one core to many cores to many nodes.** You write the same array expression; Dask ships the chunks to whatever workers are connected.

---

### Session recap

You now have a toolkit for going from slow Python to a distributed computation on Expanse:

1. **Numba `@jit`** — compile a hot loop, get 10–100x on a single core. (See `3_numba/`.)
2. **Numba `@jit(parallel=True)` and `prange`** — spread that loop across cores on one node. (See `3_numba/2_threads.ipynb`.)
3. **Dask arrays** — chunk a NumPy array and scale from one core to many cores to **many nodes** without changing the API. (See `5_dask/`.)

The artifacts you are taking with you:

* The `dask_slurm/` scripts — a working template for launching a Dask distributed cluster on Expanse.
* The Singularity image (`dask-numba-si26.sif`) and the `environment.yaml` — a reproducible Python HPC environment you can rebuild on any system with Singularity/Apptainer.
* This notebook set — reference material you can rerun on your own allocation.